# CubbyLLM 0050: the output head's share, and the shortlist's build states

Stage 1 of order 0050, on an A100 (any CUDA card works; the order asks for the A100):
1. **Decode:** one decode step of `base450m_cont` against its 128k head alone, a full 32k head and K gathered rows,
   batch 1 and 16, fp32 (~3 min).
2. **Training:** a forward + backward with the 128k head and its cross-entropy against the trunk alone, for the 450M
   and the grown 1.36B shape, time and peak memory (~5 min).
3. **Held-in states:** 64 windows of 1,024 tokens from each of the 13 shards' training region: the state the head
   reads, the full head's top-16, its log-sum-exp and the target's logit (~852k positions, ~20 min, ~1.7 GB on Drive).

Everything lands in `MyDrive/cubbyllm/head_shortlist/`. Claude Code builds and reads the index from there at home
(CPU). **Disconnect the runtime when the last cell ends.**

In [ ]:
import os, shutil, subprocess, sys
from google.colab import drive; drive.mount('/content/drive')
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!cd /content/CubbyLLM && git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
DRIVE = '/content/drive/MyDrive/cubbyllm'
OUTD = f'{DRIVE}/head_shortlist'; os.makedirs(OUTD, exist_ok=True)
CK450 = '/content/base450m_cont_final.pt'
if not os.path.exists(CK450):                      # Drive-FUSE reads are slow: one copy to the local disk
    shutil.copy(f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt', CK450)
CKGROWN = f'{DRIVE}/runs/grow_mem_grown/grow_mem_grown_final.pt'     # its shape only (weights left at init)
CACHE = f'{DRIVE}/token_cache_cont'

def stream(*args):
    p = subprocess.Popen([sys.executable, '-u', 'validation/head_share_torch.py', *args], cwd='/content/CubbyLLM',
                         text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0

def keep(mode):
    import glob
    for f in glob.glob(f'/content/CubbyLLM/validation/logs/head_share_{mode}_cuda_*.json'):
        shutil.copy(f, OUTD); print('kept', os.path.basename(f))

In [ ]:
# --- 1. decode: the head's share of a step ---
stream('decode', '--ckpt', CK450)
keep('decode')

In [ ]:
# --- 2. training: the head + cross-entropy's share of a step, the 450M and the grown 1.36B ---
stream('train', '--ckpt', CK450, '--ckpt', CKGROWN, '--micro', '8', '--seq', '1024')
keep('train')

In [ ]:
# --- 3. held-in states for the shortlist's build (the training region only; never the held-out tails) ---
stream('dump', '--ckpt', CK450, '--cache', CACHE, '--split', 'heldin', '--windows', '64', '--seq', '1024',
       '--name', 'heldin_w64', '--out', '/content/head_shortlist')
shutil.copy('/content/head_shortlist/heldin_w64.npz', OUTD)
keep('dump')
print('done: disconnect the runtime')